# TigrinyaLens: Testing English-to-Tigrinya Translation Quality

**Track:** AI + Languages  
**Starting point:** Section 4.4.2, Translation Quality Test  
**Model:** `facebook/nllb-200-distilled-600M`  
**Dataset:** `openlanguagedata/flores_plus`  
**Languages:** English → Tigrinya and English → French  
**Test sample:** 100 matching English sentences from `devtest`

## What the project does

TigrinyaLens compares translations of the same 100 English sentences into Tigrinya and French. It measures similarity to professional FLORES+ references using **chrF** (the primary metric) and **BLEU** (secondary metric). It also identifies five low-scoring examples per language and prepares a short human-review task for the three Tigrinya-speaking team members.

**Input:** an English sentence.  
**Output:** a machine-generated Tigrinya translation and a French translation, plus automatic evaluation scores across the test set.

**Intended user:** a Tigrinya-speaking secondary-school student in Tigray who receives school or educational information in English and wants to understand it in Tigrinya. The tool is an experimental prototype; it does not guarantee correct translations and must not be relied on for medical, legal, or other high-stakes decisions.

## Research rules

- This notebook uses a pretrained model for inference. It does not train or fine-tune NLLB.
- It uses the FLORES+ `devtest` split as held-out evaluation data.
- The same English sentence IDs are used for both target languages.
- The English-copy baseline is a negative control, not a competing translation model.
- A low chrF score is a prompt to inspect a sentence, not proof that the translation is wrong.
- Human ratings must be completed independently by all three team members. Never invent ratings or conclusions.

**Fresh Colab setup:** use a new Google Colab notebook, select a GPU if available, and run these cells from top to bottom. If a cell fails, stop at that cell and share the complete error; do not silently change the dataset or variable names.

## 1. Install the libraries

Run once in the new Colab runtime. Colab already provides PyTorch in most runtimes, so this cell does not reinstall it.

In [ ]:
%pip -q install -U datasets transformers sentencepiece sacrebleu huggingface_hub pandas numpy matplotlib scipy gradio

## 2. Import libraries

In [ ]:
import os
import json
import random
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import sacrebleu

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from IPython.display import display

print("Imports completed.")
print("PyTorch:", torch.__version__)
print("SacreBLEU:", sacrebleu.__version__)

## 3. Hugging Face access (login is normally optional)

The FLORES+ dataset and NLLB-200 model are public Hugging Face resources, so authentication is normally not required. If a download returns an authorization error, sign in to Hugging Face through its secure login flow in the Colab session. Never paste an access token into a code cell, notebook output, chat, or public repository.

## 4. Set the experiment configuration

Only replace the team placeholders. The language codes and dataset/model identifiers are fixed for this experiment.

In [ ]:
# Public project team details. Contact emails are intentionally omitted.
TEAM_NAME = "GEREX"
TEAM_MEMBERS = [
    {"name": "Amaniel Mebrahtom", "school": "Kallamino Special High School"},
    {"name": "Ambassador Girmay", "school": "Kallamino Special High School"},
    {"name": "Abenezer Shebabaw", "school": "Kallamino Special High School"},
]

# Dataset and model
DATASET_ID = "openlanguagedata/flores_plus"
MODEL_ID = "facebook/nllb-200-distilled-600M"

# NLLB language codes
SOURCE_LANG = "eng_Latn"
TIGRINYA_LANG = "tir_Ethi"
FRENCH_LANG = "fra_Latn"

# Experiment settings
SPLIT_NAME = "devtest"
NUMBER_OF_TEST_EXAMPLES = 100
NUMBER_OF_HUMAN_EVALUATION_EXAMPLES = 10
RANDOM_SEED = 42
BATCH_SIZE = 8
MAX_SOURCE_LENGTH = 256
MAX_NEW_TOKENS = 256
BEAM_SIZE = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

OUTPUT_DIR = "/content/tigrinyalens_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project: TigrinyaLens")
print("Team:", TEAM_NAME)
print("Dataset:", DATASET_ID)
print("Model:", MODEL_ID)
print("Device:", DEVICE)
print("Test sentences:", NUMBER_OF_TEST_EXAMPLES)

## 5. Load the three FLORES+ language configurations

Your actual FLORES+ repository output showed separate configurations named `eng_Latn`, `tir_Ethi`, and `fra_Latn`, rather than paired configuration names. Each language split uses the columns `id` and `text`. This cell follows that observed structure.

In [ ]:
print("Loading English...")
english_data = load_dataset(DATASET_ID, SOURCE_LANG)

print("Loading Tigrinya...")
tigrinya_data = load_dataset(DATASET_ID, TIGRINYA_LANG)

print("Loading French...")
french_data = load_dataset(DATASET_ID, FRENCH_LANG)

for language, dataset_dict in [
    ("English", english_data),
    ("Tigrinya", tigrinya_data),
    ("French", french_data),
]:
    if SPLIT_NAME not in dataset_dict:
        raise ValueError(
            f"'{SPLIT_NAME}' split not found for {language}. "
            f"Available splits: {list(dataset_dict.keys())}"
        )

english_test = english_data[SPLIT_NAME]
tigrinya_test = tigrinya_data[SPLIT_NAME]
french_test = french_data[SPLIT_NAME]

for language, split in [
    ("English", english_test),
    ("Tigrinya", tigrinya_test),
    ("French", french_test),
]:
    if not {"id", "text"}.issubset(split.column_names):
        raise ValueError(
            f"Unexpected {language} columns: {split.column_names}. "
            "This notebook expects the observed FLORES+ schema: id and text."
        )

print("FLORES+ language datasets loaded.")
print("English rows:", len(english_test))
print("Tigrinya rows:", len(tigrinya_test))
print("French rows:", len(french_test))

## 6. Align the languages and select 100 shared sentences

We align by sentence ID, not by assuming row order. The same selected IDs are used for English, Tigrinya, and French. The fixed random seed makes the sample reproducible.

In [ ]:
def make_id_text_map(split, language_name):
    mapping = {}
    for row in split:
        sentence_id = str(row["id"])
        text = row["text"]
        if sentence_id in mapping:
            raise ValueError(f"Duplicate sentence ID in {language_name}: {sentence_id}")
        if not isinstance(text, str) or not text.strip():
            raise ValueError(f"Empty or invalid sentence in {language_name}: {sentence_id}")
        mapping[sentence_id] = text.strip()
    return mapping

english_map = make_id_text_map(english_test, "English")
tigrinya_map = make_id_text_map(tigrinya_test, "Tigrinya")
french_map = make_id_text_map(french_test, "French")

shared_ids = sorted(set(english_map) & set(tigrinya_map) & set(french_map))
if len(shared_ids) < NUMBER_OF_TEST_EXAMPLES:
    raise ValueError(
        f"Only {len(shared_ids)} sentence IDs are shared across the three languages; "
        f"{NUMBER_OF_TEST_EXAMPLES} are required."
    )

rng = random.Random(RANDOM_SEED)
example_ids = rng.sample(shared_ids, NUMBER_OF_TEST_EXAMPLES)

english_sentences = [english_map[sid] for sid in example_ids]
tigrinya_references = [tigrinya_map[sid] for sid in example_ids]
french_references = [french_map[sid] for sid in example_ids]

assert len(english_sentences) == NUMBER_OF_TEST_EXAMPLES
assert len(tigrinya_references) == NUMBER_OF_TEST_EXAMPLES
assert len(french_references) == NUMBER_OF_TEST_EXAMPLES

print("Paired data verified.")
print("English:", len(english_sentences))
print("Tigrinya references:", len(tigrinya_references))
print("French references:", len(french_references))
print("Example ID:", example_ids[0])
print("\nEnglish:", english_sentences[0])
print("Tigrinya reference:", tigrinya_references[0])
print("French reference:", french_references[0])

## 7. Load NLLB-200

The model is used for inference only. The notebook checks that the model tokenizer recognizes all three language codes.

In [ ]:
print("Loading NLLB tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    src_lang=SOURCE_LANG
)
tokenizer.src_lang = SOURCE_LANG

print("Loading NLLB model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
model = model.to(DEVICE)
model.eval()

FRENCH_TOKEN_ID = tokenizer.convert_tokens_to_ids(FRENCH_LANG)
TIGRINYA_TOKEN_ID = tokenizer.convert_tokens_to_ids(TIGRINYA_LANG)

if FRENCH_TOKEN_ID == tokenizer.unk_token_id:
    raise ValueError("NLLB tokenizer does not recognize fra_Latn.")
if TIGRINYA_TOKEN_ID == tokenizer.unk_token_id:
    raise ValueError("NLLB tokenizer does not recognize tir_Ethi.")

print("Model loaded on:", DEVICE)
print("French token ID:", FRENCH_TOKEN_ID)
print("Tigrinya token ID:", TIGRINYA_TOKEN_ID)

## 8. Define one translation function

Greedy decoding (`num_beams=1`) and beam search (`num_beams=4`) are two decoding strategies for the same model, not two different models. We use deterministic decoding so repeated runs with the same setup are comparable.

In [ ]:
def translate_batch(texts, target_language, num_beams=1):
    if target_language not in [TIGRINYA_LANG, FRENCH_LANG]:
        raise ValueError(f"Unsupported target language: {target_language}")
    if not texts:
        return []

    tokenizer.src_lang = SOURCE_LANG
    target_token_id = tokenizer.convert_tokens_to_ids(target_language)

    encoded = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
    )
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            forced_bos_token_id=target_token_id,
            num_beams=num_beams,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
        )

    return tokenizer.batch_decode(generated, skip_special_tokens=True)

# Test both target languages before processing all 100 examples.
print("Tigrinya test:", translate_batch([english_sentences[0]], TIGRINYA_LANG, BEAM_SIZE)[0])
print("French test:", translate_batch([english_sentences[0]], FRENCH_LANG, BEAM_SIZE)[0])

## 9. Generate the translations

This creates four output lists: French greedy, French beam-4, Tigrinya greedy, and Tigrinya beam-4.

In [ ]:
tigrinya_greedy = []
tigrinya_beam = []
french_greedy = []
french_beam = []

for start in range(0, len(english_sentences), BATCH_SIZE):
    batch = english_sentences[start:start + BATCH_SIZE]

    tigrinya_greedy.extend(translate_batch(batch, TIGRINYA_LANG, num_beams=1))
    tigrinya_beam.extend(translate_batch(batch, TIGRINYA_LANG, num_beams=BEAM_SIZE))
    french_greedy.extend(translate_batch(batch, FRENCH_LANG, num_beams=1))
    french_beam.extend(translate_batch(batch, FRENCH_LANG, num_beams=BEAM_SIZE))

    print(f"Translated {min(start + BATCH_SIZE, len(english_sentences))}/{len(english_sentences)} source sentences")

for name, outputs in [
    ("Tigrinya greedy", tigrinya_greedy),
    ("Tigrinya beam", tigrinya_beam),
    ("French greedy", french_greedy),
    ("French beam", french_beam),
]:
    if len(outputs) != NUMBER_OF_TEST_EXAMPLES:
        raise RuntimeError(f"Unexpected output count for {name}: {len(outputs)}")

print("All translation runs completed.")

## 10. Inspect sample translations

Check that the outputs are in the intended languages and contain text. Fluency must still be checked by people; a non-empty output is not proof of correctness.

In [ ]:
sample_table = pd.DataFrame({
    "English": english_sentences[:5],
    "Tigrinya model": tigrinya_beam[:5],
    "Tigrinya reference": tigrinya_references[:5],
    "French model": french_beam[:5],
    "French reference": french_references[:5],
})
display(sample_table)

## 11. Normalize the text

This applies Unicode NFC normalization and trims repeated whitespace. It does not alter sentence meaning or repair translation errors.

In [ ]:
def normalize_text(text):
    return " ".join(__import__("unicodedata").normalize("NFC", str(text)).split())

english_norm = [normalize_text(x) for x in english_sentences]
tigrinya_ref_norm = [normalize_text(x) for x in tigrinya_references]
french_ref_norm = [normalize_text(x) for x in french_references]
tigrinya_greedy_norm = [normalize_text(x) for x in tigrinya_greedy]
tigrinya_beam_norm = [normalize_text(x) for x in tigrinya_beam]
french_greedy_norm = [normalize_text(x) for x in french_greedy]
french_beam_norm = [normalize_text(x) for x in french_beam]

print("Normalization completed.")

## 12. Calculate chrF and BLEU

chrF is the primary metric requested by the hackathon brief. BLEU is reported as a secondary metric. These are similarity metrics, not percentages of translation accuracy.

In [ ]:
def calculate_metrics(predictions, references):
    if len(predictions) != len(references):
        raise ValueError("Prediction and reference lengths do not match.")
    return {
        "chrF": sacrebleu.corpus_chrf(predictions, [references]).score,
        "BLEU": sacrebleu.corpus_bleu(predictions, [references]).score,
    }

metric_rows = []
conditions = [
    ("Tigrinya", "Greedy decoding", tigrinya_greedy_norm, tigrinya_ref_norm),
    ("Tigrinya", "Beam search (4 beams)", tigrinya_beam_norm, tigrinya_ref_norm),
    ("French", "Greedy decoding", french_greedy_norm, french_ref_norm),
    ("French", "Beam search (4 beams)", french_beam_norm, french_ref_norm),
]

for language, method, predictions, references in conditions:
    scores = calculate_metrics(predictions, references)
    metric_rows.append({
        "Language": language,
        "Method": method,
        "Examples": len(references),
        "chrF": scores["chrF"],
        "BLEU": scores["BLEU"],
    })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(2))

## 13. Compare against a simple copy-source negative control

This baseline returns the English input unchanged and scores it against the French or Tigrinya reference. It shows the result of not translating at all. It is a deliberately weak negative control, not a competing translation system.

In [ ]:
copy_baseline_tigrinya = calculate_metrics(english_norm, tigrinya_ref_norm)
copy_baseline_french = calculate_metrics(english_norm, french_ref_norm)

baseline_df = pd.DataFrame([
    {"Language": "Tigrinya", "Method": "English-copy negative control", **copy_baseline_tigrinya},
    {"Language": "French", "Method": "English-copy negative control", **copy_baseline_french},
])

all_metrics = pd.concat([baseline_df, metrics_df], ignore_index=True)
print("All language and method results:")
display(all_metrics.round(2))

## 14. Identify the five lowest-scoring translations in each language

We rank beam-search outputs by sentence-level chrF. These are **low-scoring candidates for manual inspection**, not automatically proven translation errors. A valid alternative translation can have low overlap with a single reference.

In [ ]:
def make_sentence_analysis(language, sources, hypotheses, references, ids):
    rows = []
    for sentence_id, source, hypothesis, reference in zip(ids, sources, hypotheses, references):
        sentence_chrf = sacrebleu.sentence_chrf(hypothesis, [reference]).score
        sentence_bleu = sacrebleu.sentence_bleu(hypothesis, [reference]).score
        rows.append({
            "example_id": sentence_id,
            "language": language,
            "English": source,
            "Model output": hypothesis,
            "Professional reference": reference,
            "sentence_chrF": sentence_chrf,
            "sentence_BLEU": sentence_bleu,
            "verified_error_category": "",
            "team_explanation": "",
        })
    return pd.DataFrame(rows)

tigrinya_analysis = make_sentence_analysis(
    "Tigrinya", english_norm, tigrinya_beam_norm, tigrinya_ref_norm, example_ids
)
french_analysis = make_sentence_analysis(
    "French", english_norm, french_beam_norm, french_ref_norm, example_ids
)

tigrinya_worst5 = tigrinya_analysis.nsmallest(5, "sentence_chrF").reset_index(drop=True)
french_worst5 = french_analysis.nsmallest(5, "sentence_chrF").reset_index(drop=True)

print("Five lowest-scoring Tigrinya examples:")
display(tigrinya_worst5)
print("Five lowest-scoring French examples:")
display(french_worst5)

## 15. Prepare a short, blinded human-evaluation task

The three team members independently rate the **same 10 NLLB-200 Tigrinya model outputs** (the beam-search outputs). They do **not** rate the FLORES+ reference. They judge the model output against the English source. The professional FLORES+ reference and automatic chrF score are deliberately hidden from the raters to avoid influencing their judgments. After all ratings are submitted, the notebook joins the ratings to the reference and chrF for analysis.

Rate each NLLB Tigrinya output from 1 to 5 for:
- **Meaning:** Does the Tigrinya output preserve the English meaning?
- **Fluency:** Does it sound natural and understandable in Tigrinya?
- **Grammar:** Is the Tigrinya grammatically acceptable?

Use an error category if relevant: `no_obvious_error`, `incorrect_meaning`, `omission`, `grammar`, `names_numbers`, `unnatural_wording`, or `other`. Rate independently. Do not discuss scores until all three files are complete.

In [ ]:
from google.colab import files

# Select the same reproducible 10 Tigrinya beam-search outputs for all raters.
human_rng = random.Random(RANDOM_SEED + 1)
human_positions = sorted(
    human_rng.sample(
        range(NUMBER_OF_TEST_EXAMPLES),
        NUMBER_OF_HUMAN_EVALUATION_EXAMPLES
    )
)

# Keep reference and chrF internally for analysis AFTER ratings are completed.
human_base = pd.DataFrame({
    "example_id": [str(example_ids[i]) for i in human_positions],
    "English": [english_sentences[i] for i in human_positions],
    "Tigrinya model output": [tigrinya_beam[i] for i in human_positions],
    "FLORES+ reference (hidden from raters)": [tigrinya_references[i] for i in human_positions],
    "sentence_chrF (hidden from raters)": [
        tigrinya_analysis.iloc[i]["sentence_chrF"] for i in human_positions
    ],
})

# Raters receive only the English source and the NLLB model output.
rater_template = human_base[[
    "example_id",
    "English",
    "Tigrinya model output"
]].copy()

RATER_FILES = ["rater_1.csv", "rater_2.csv", "rater_3.csv"]

for filename in RATER_FILES:
    sheet = rater_template.copy()
    sheet["Meaning (1-5)"] = ""
    sheet["Fluency (1-5)"] = ""
    sheet["Grammar (1-5)"] = ""
    sheet["Error category"] = ""
    sheet["Comment"] = ""
    sheet.to_csv(
        os.path.join(OUTPUT_DIR, filename),
        index=False,
        encoding="utf-8-sig"
    )

instructions = """TIGRINYALENS — SHORT, BLINDED HUMAN EVALUATION

WHAT TO RATE: Rate the Tigrinya MODEL OUTPUT produced by NLLB-200 (beam search). Do NOT rate the FLORES+ reference. The reference and automatic chrF score are intentionally not included in your file.

Read the English source, then judge whether the Tigrinya model output preserves its meaning and is natural and grammatical. Different valid wording is acceptable.

Complete the same 10 examples independently. Do not discuss ratings with the other evaluators until all three files are complete.

Use whole-number scores from 1 to 5:
Meaning: 1 = meaning mostly wrong or missing; 3 = partly correct; 5 = meaning faithfully preserved.
Fluency: 1 = difficult to understand; 3 = understandable but unnatural; 5 = natural and clear.
Grammar: 1 = serious grammatical problems; 3 = noticeable problems; 5 = grammatical and clear.

Error categories: no_obvious_error, incorrect_meaning, omission, grammar, names_numbers, unnatural_wording, other.
Add a short comment when it helps explain a problem. Do not edit example_id, English, or Tigrinya model output. Save your completed file with its original filename.
"""

with open(
    os.path.join(OUTPUT_DIR, "human_evaluation_instructions.txt"),
    "w",
    encoding="utf-8"
) as f:
    f.write(instructions)

# Download the three blank, separate rating sheets and instructions.
template_zip = os.path.join(OUTPUT_DIR, "human_evaluation_templates.zip")
with zipfile.ZipFile(template_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in RATER_FILES:
        z.write(os.path.join(OUTPUT_DIR, filename), arcname=filename)
    z.write(
        os.path.join(OUTPUT_DIR, "human_evaluation_instructions.txt"),
        arcname="human_evaluation_instructions.txt"
    )

print("Created three independent, blinded evaluation sheets.")
print("Raters evaluate NLLB Tigrinya model output, NOT the FLORES+ reference.")
files.download(template_zip)

## 16. Upload the three completed human-evaluation files

Run this cell only after all three team members have independently completed their rating sheets. Upload the files named exactly `rater_1.csv`, `rater_2.csv`, and `rater_3.csv`. Each sheet must contain ratings for the same 10 NLLB Tigrinya model outputs. The raters should not have seen the FLORES+ reference or chrF score while rating.

In [ ]:
uploaded = files.upload()
missing = set(RATER_FILES) - set(uploaded.keys())
if missing:
    raise FileNotFoundError("Upload all three completed files. Missing: " + ", ".join(sorted(missing)))
print("All three completed rating files uploaded.")

## 17. Validate and summarize the human ratings

This cell checks that each rater scored the same 10 example IDs and entered whole-number ratings from 1 to 5. It then calculates the mean human score per example and the Spearman correlation with sentence-level chrF. With only 10 examples and three team raters, interpret the correlation as exploratory only.

In [ ]:
expected_ids = set(human_base["example_id"].astype(str))
rating_columns = ["Meaning (1-5)", "Fluency (1-5)", "Grammar (1-5)"]
all_ratings = []

for rater_number, filename in enumerate(RATER_FILES, start=1):
    df = pd.read_csv(filename, dtype={"example_id": str})
    required = {"example_id", *rating_columns, "Error category", "Comment"}
    if not required.issubset(df.columns):
        raise ValueError(
            f"{filename} is missing columns: {sorted(required - set(df.columns))}"
        )

    if df["example_id"].duplicated().any() or set(df["example_id"]) != expected_ids:
        raise ValueError(
            f"Example IDs in {filename} do not match the human-evaluation sample."
        )

    for column in rating_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")
        if df[column].isna().any() or not df[column].between(1, 5).all():
            raise ValueError(
                f"{filename}: {column} must contain complete ratings from 1 to 5."
            )
        if not np.equal(df[column], np.floor(df[column])).all():
            raise ValueError(
                f"{filename}: {column} must contain whole-number ratings."
            )

    df["Rater"] = f"Rater {rater_number}"
    all_ratings.append(df)

ratings_long = pd.concat(all_ratings, ignore_index=True)

ratings_summary = (
    ratings_long.groupby("example_id")[rating_columns]
    .mean()
    .reset_index()
)
ratings_summary["Mean human rating"] = ratings_summary[rating_columns].mean(axis=1)

# Reveal/attach the professional reference and chrF only AFTER blind ratings are complete.
human_analysis = human_base.merge(
    ratings_summary,
    on="example_id",
    validate="one_to_one"
)

human_analysis["sentence_chrF"] = pd.to_numeric(
    human_analysis["sentence_chrF (hidden from raters)"],
    errors="raise"
)

correlations = []
for column in rating_columns:
    if (
        human_analysis[column].nunique() < 2
        or human_analysis["sentence_chrF"].nunique() < 2
    ):
        rho, p_value = np.nan, np.nan
    else:
        correlation = spearmanr(
            human_analysis["sentence_chrF"],
            human_analysis[column]
        )
        rho = float(correlation.statistic)
        p_value = float(correlation.pvalue)

    correlations.append({
        "Human rating dimension": column,
        "Spearman rho": rho,
        "p-value": p_value,
        "n": len(human_analysis),
    })

correlation_df = pd.DataFrame(correlations)

print("Average human ratings:")
display(
    human_analysis[rating_columns + ["Mean human rating"]]
    .mean()
    .to_frame("Average score")
    .round(2)
)

print("Exploratory correlation with sentence-level chrF:")
display(correlation_df.round(3))

ratings_long.to_csv(
    os.path.join(OUTPUT_DIR, "human_ratings_all.csv"),
    index=False,
    encoding="utf-8-sig"
)
human_analysis.to_csv(
    os.path.join(OUTPUT_DIR, "human_ratings_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)
correlation_df.to_csv(
    os.path.join(OUTPUT_DIR, "human_chrf_correlation.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("Human evaluation analysis completed.")

## 18. Save metrics, charts, metadata, and a write-up draft

In [ ]:
# Save automatic metrics and comparison table.
metrics_path = os.path.join(OUTPUT_DIR, "translation_metrics.csv")
all_metrics_path = os.path.join(OUTPUT_DIR, "translation_metrics_with_baseline.csv")
metrics_df.to_csv(metrics_path, index=False)
all_metrics.to_csv(all_metrics_path, index=False)

# Save sentence-level data and worst examples.
results_df = pd.DataFrame({
    "example_id": example_ids,
    "English": english_sentences,
    "Tigrinya reference": tigrinya_references,
    "Tigrinya greedy": tigrinya_greedy,
    "Tigrinya beam": tigrinya_beam,
    "French reference": french_references,
    "French greedy": french_greedy,
    "French beam": french_beam,
})
results_df.to_csv(os.path.join(OUTPUT_DIR, "all_translations.csv"), index=False, encoding="utf-8-sig")
tigrinya_worst5.to_csv(os.path.join(OUTPUT_DIR, "tigrinya_five_lowest_chrf.csv"), index=False, encoding="utf-8-sig")
french_worst5.to_csv(os.path.join(OUTPUT_DIR, "french_five_lowest_chrf.csv"), index=False, encoding="utf-8-sig")

# Automatic chrF chart.
chart_path = os.path.join(OUTPUT_DIR, "chrf_comparison.png")
chart_data = all_metrics.pivot(index="Language", columns="Method", values="chrF")
ax = chart_data.plot(kind="bar", figsize=(10, 5))
ax.set_title("NLLB-200 translation evaluation by target language")
ax.set_ylabel("Corpus chrF")
ax.set_xlabel("Target language")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

# Metadata for reproducibility.
metadata = {
    "project": PROJECT_TITLE,
    "team_name": TEAM_NAME,
    "team_members": TEAM_MEMBERS,
    "dataset_id": DATASET_ID,
    "tigrinya_configuration": TIGRINYA_LANG,
    "french_configuration": FRENCH_LANG,
    "split": SPLIT_NAME,
    "model": MODEL_ID,
    "source_language": SOURCE_LANG,
    "tigrinya_language": TIGRINYA_LANG,
    "french_language": FRENCH_LANG,
    "number_of_test_examples": NUMBER_OF_TEST_EXAMPLES,
    "random_seed": RANDOM_SEED,
    "decoding_methods": ["greedy (num_beams=1)", f"beam search (num_beams={BEAM_SIZE})"],
    "device": DEVICE,
    "pytorch_version": str(torch.__version__),
    "datasets_version": __import__("datasets").__version__,
    "transformers_version": __import__("transformers").__version__,
    "sacrebleu_version": str(sacrebleu.__version__),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
with open(os.path.join(OUTPUT_DIR, "experiment_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# Draft write-up with measured automatic scores; human findings remain for the team to complete.
metric_lines = []
for _, row in all_metrics.iterrows():
    metric_lines.append(
        f"| {row['Language']} | {row['Method']} | {row['chrF']:.2f} | {row['BLEU']:.2f} |"
    )

writeup = f"""# TigrinyaLens: Testing English-to-Tigrinya Translation Quality\n\n**Track:** AI + Languages  \n**Team:** {TEAM_NAME} — [enter all three names]  \n**Dataset:** {DATASET_ID}, {SPLIT_NAME} split  \n**Model:** {MODEL_ID} (pretrained inference; no fine-tuning)\n\n## The problem\nTigrinya speakers may rely on machine translation to understand information originally written in English. A fluent-looking output can still change or omit meaning. We investigate the translation quality of one multilingual model and whether an automatic character-overlap metric reflects judgments by Tigrinya speakers.\n\n## What we built\nTigrinyaLens translates the same {NUMBER_OF_TEST_EXAMPLES} English sentences into Tigrinya and French using NLLB-200. We compare greedy decoding and beam-{BEAM_SIZE} decoding against professional FLORES+ references. The intended user is a Tigrinya-speaking student or community member seeking to understand English text. This is an evaluation prototype, not a guarantee of translation correctness.\n\n## What we added\nThree Tigrinya-speaking team members independently review {NUMBER_OF_HUMAN_EVALUATION_EXAMPLES} Tigrinya translations for meaning, fluency, and grammar. We compare the average human scores with sentence-level chrF. We also inspect the five lowest-chrF examples in each language and manually verify whether they contain actual errors.\n\n## How it works\n**Data:** {NUMBER_OF_TEST_EXAMPLES} paired English sentences and corresponding French and Tigrinya references from the {SPLIT_NAME} split of {DATASET_ID}.  \n**Model:** {MODEL_ID}.  \n**Evaluation:** chrF (primary), BLEU (secondary), greedy decoding, beam search, and an English-copy negative control.\n\n## Results\n| Language | Method | chrF | BLEU |\n|---|---|---:|---:|\n{chr(10).join(metric_lines)}\n\n**Human evaluation:** [Insert the actual three-rater averages and exploratory correlation results after completing the rating sheets.]\n\n**Verified failure example:** [Insert an English source, model output, professional reference, and the team’s explanation. Do not label a low chrF score alone as a translation error.]\n\n## What we would do next\nEvaluate more sentences, involve additional independent Tigrinya speakers, inspect more domains, and test additional translation models.\n\n## Sources\n- FLORES+: https://huggingface.co/datasets/openlanguagedata/flores_plus\n- NLLB-200 model: https://huggingface.co/facebook/nllb-200-distilled-600M\n- Hugging Face Datasets and Transformers; PyTorch; SacreBLEU; pandas; NumPy; Matplotlib; SciPy; Gradio.\n- AI assistance: ChatGPT and Gemini were used for project planning, notebook drafting, and debugging guidance. The team reviewed and ran the code and must accurately describe its own use of AI tools.\n"""
with open(os.path.join(OUTPUT_DIR, "one_page_writeup_draft.md"), "w", encoding="utf-8") as f:
    f.write(writeup)

print("Core results, chart, metadata, and write-up draft saved in:", OUTPUT_DIR)

## 19. Optional interactive Gradio demo

The demo accepts an English sentence and returns French and Tigrinya model outputs. It is not a quality checker. The default is a local notebook demo; change `SHARE_DEMO` to `True` only if you need a temporary public link. Anyone with a public link may access it while it is active, so do not enter private text.

In [ ]:
import gradio as gr

SHARE_DEMO = False  # Change to True only if you need a temporary public link.

def demo_translate(text):
    if text is None or not str(text).strip():
        return "Enter an English sentence.", "Enter an English sentence."
    french_output = translate_batch([str(text).strip()], FRENCH_LANG, num_beams=BEAM_SIZE)[0]
    tigrinya_output = translate_batch([str(text).strip()], TIGRINYA_LANG, num_beams=BEAM_SIZE)[0]
    return french_output, tigrinya_output

demo = gr.Interface(
    fn=demo_translate,
    inputs=gr.Textbox(lines=3, label="English input"),
    outputs=[gr.Textbox(label="French translation"), gr.Textbox(label="Tigrinya translation")],
    title="TigrinyaLens",
    description="Experimental NLLB-200 translations. Outputs may contain errors and are not verified translations."
)
demo.launch(share=SHARE_DEMO)

## 20. Package the project outputs

Run after the translation and metric sections. If you have completed the human ratings, the human analysis files will also be included automatically.

In [ ]:
zip_path = "/content/TigrinyaLens_results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for root, _, filenames in os.walk(OUTPUT_DIR):
        for filename in filenames:
            full_path = os.path.join(root, filename)
            relative_path = os.path.relpath(full_path, OUTPUT_DIR)
            archive.write(full_path, arcname=os.path.join("TigrinyaLens", relative_path))

print("Created:", zip_path)
print("Included files:")
for root, _, filenames in os.walk(OUTPUT_DIR):
    for filename in sorted(filenames):
        print("-", os.path.relpath(os.path.join(root, filename), OUTPUT_DIR))

from google.colab import files
files.download(zip_path)

## Reproducibility and reporting checklist

Before sharing a new run of this notebook, verify the following:

1. Confirm the loaded dataset is `openlanguagedata/flores_plus`, with the `eng_Latn`, `tir_Ethi`, and `fra_Latn` configurations and the `devtest` split.
2. Confirm 100 identical sentence IDs are used across all three languages.
3. Check that each of the four translation output lists contains 100 examples.
4. Report chrF and BLEU as similarity metrics, never as accuracy percentages.
5. Treat the lowest-chrF examples as candidates for manual review, not automatically verified errors.
6. Use only the actual independent ratings submitted by the three evaluators. Do not invent or retrospectively alter ratings.
7. Describe the human-review sample as small and exploratory.
8. Keep model/data versions, environment details, and generated results with the run when practical.
9. Never commit access tokens, private contact details, or unreviewed personal information to this public repository.

The notebook checks data structure and expected output counts, but cannot guarantee that Colab, Hugging Face, GPU availability, or model downloads will never encounter a temporary error.